![Banner Proposições](https://www.camara.leg.br/internet/deputado/img/logo-camara.png)

# 📜 Fast Track — 07. Bronze: Proposições

**Ingestão de Dados Brutos — API Câmara dos Deputados**

| Item | Descrição |
|---|---|
| **Objetivo** | Ingerir proposições legislativas (PLs, PECs, MPs, etc) |
| **Entidade** | Proposições |
| **Endpoint API** | `https://dadosabertos.camara.leg.br/api/v2/proposicoes` |
| **Tabela Destino** | `uc_fast_track.bronze.proposicoes_raw` |
| **Modo** | APPEND |
| **Checkpoint** | Por `id` da proposição |
| **Formato** | Payload JSON raw + colunas técnicas |

---

## 🎯 O Que São Proposições?

**Proposições** são todas as matérias legislativas que tramitam na Câmara:

**Tipos principais:**
* **PL** (Projeto de Lei): Propõe criação, alteração ou revogação de leis
* **PEC** (Proposta de Emenda à Constituição): Modifica a Constituição
* **MP** (Medida Provisória): Ato do Executivo com força de lei
* **PLP** (Projeto de Lei Complementar): Leis que complementam a Constituição
* **PDL** (Projeto de Decreto Legislativo): Assuntos de competência exclusiva do Congresso

**Características:**
* Tramitam por comissões
* São votadas em plenário
* Podem receber emendas
* Têm autores (deputados, senadores, Executivo)
* Status: Em tramitação, Aprovada, Arquivada, etc

## ⚙️ Bloco 1: Configuração

In [0]:
print("🔗 Recuperando variáveis...")
try:
    print(f"✅ Load ID: {load_id}")
    print(f"✅ Catálogo: {catalog}")
except:
    raise Exception("Execute notebook 00")

entity_name = "proposicoes"
api_base_url = "https://dadosabertos.camara.leg.br/api/v2"
api_endpoint = f"{api_base_url}/proposicoes"
table_bronze = f"{catalog}.{schema_bronze}.{entity_name}_raw"
page_size = 100
max_retries = 3

print(f"\n✅ Entidade: {entity_name}")
print(f"✅ Endpoint: {api_endpoint}")
print(f"✅ Tabela: {table_bronze}")
print("="*70)

## 📦 Bloco 2: Importações

In [0]:
import requests, json, time, uuid, hashlib
from datetime import datetime
from pyspark.sql.types import *
print("✅ OK")

## 🔌 Bloco 3: Função Extração

In [0]:
def get_proposicoes_from_api(checkpoint_id=None):
    print(f"🔌 Extraindo de: {api_endpoint}")
    all_data = []
    pagina = 1
    has_more = True
    total_reqs = 0
    
    while has_more:
        print(f"📄 Página {pagina}...")
        params = {"pagina": pagina, "itens": page_size, "ordem": "ASC", "ordenarPor": "id"}
        req_id = str(uuid.uuid4())
        start = time.time()
        success = False
        resp_data = None
        status = None
        err = None
        
        for attempt in range(1, max_retries + 1):
            try:
                r = requests.get(api_endpoint, params=params, timeout=30)
                status = r.status_code
                if r.status_code == 200:
                    resp_data = r.json()
                    success = True
                    break
                else:
                    err = f"HTTP {r.status_code}"
                    if attempt < max_retries:
                        time.sleep(3 * attempt)
            except Exception as e:
                err = str(e)
                if attempt < max_retries:
                    time.sleep(3 * attempt)
        
        resp_time = int((time.time() - start) * 1000)
        total_reqs += 1
        
        spark.createDataFrame([{
            "request_id": req_id, "load_id": load_id, "entity_name": entity_name,
            "endpoint": f"{api_endpoint}?pagina={pagina}", "http_method": "GET",
            "http_status": status, "response_time_ms": resp_time, "success": success,
            "error_message": err, "request_timestamp": datetime.now()
        }]).write.mode("append").saveAsTable(f"{catalog}.{schema_ops}.ingestion_requests")
        
        if not success:
            raise Exception(f"Falha: {err}")
        
        data_page = resp_data.get("dados", [])
        print(f"   ✅ {len(data_page)} registros ({resp_time}ms)")
        
        if checkpoint_id:
            data_page = [d for d in data_page if d.get("id", 0) > checkpoint_id]
            print(f"   🔍 Filtrados: {len(data_page)} novos")
        
        all_data.extend(data_page)
        
        links = resp_data.get("links", [])
        has_more = any(l.get("rel") == "next" for l in links) and len(data_page) > 0
        if has_more:
            pagina += 1
            time.sleep(0.5)
        else:
            print("   ℹ️ Última página")
        print()
    
    print(f"✅ Extração completa: {total_reqs} requests, {len(all_data)} registros")
    return all_data

print("✅ Função definida")

## 🔖 Bloco 4: Checkpoint

In [0]:
print("🔖 Checkpoint...\n")

if full_refresh:
    checkpoint_value = None
    print("🔄 FULL REFRESH")
else:
    try:
        df_ck = spark.sql(f"""
            SELECT checkpoint_value FROM {catalog}.{schema_ops}.checkpoints
            WHERE source_endpoint = '/proposicoes' AND checkpoint_type = 'last_id'
            ORDER BY updated_at DESC LIMIT 1
        """)
        checkpoint_value = int(df_ck.first()["checkpoint_value"]) if df_ck.count() > 0 else None
        print(f"✅ Checkpoint: {checkpoint_value}" if checkpoint_value else "ℹ️ Primeira exec")
    except:
        checkpoint_value = None

print("="*70)

## 🗄️ Bloco 5: Criar Tabela

In [0]:
print(f"🗄️ {table_bronze}...\n")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {table_bronze} (
    payload STRING COMMENT 'Payload JSON raw',
    record_id STRING COMMENT 'ID proposição',
    record_hash STRING COMMENT 'Hash SHA256',
    load_id STRING COMMENT 'UUID execução',
    ingestion_timestamp TIMESTAMP,
    ingestion_date DATE,
    source_endpoint STRING,
    env STRING
)
USING DELTA PARTITIONED BY (ingestion_date)
COMMENT 'Bronze - Proposições Raw (PLs, PECs, MPs, etc)'
""")

print("✅ OK")
print("="*70)

## 📥 Bloco 6: Ingestão

In [0]:
print("📥 Ingestão...\n")
ingest_start = time.time()

data_list = get_proposicoes_from_api(checkpoint_id=checkpoint_value)
num_records = len(data_list)

print(f"✅ {num_records} proposições\n")

if num_records == 0:
    spark.createDataFrame([{
        "load_id": load_id, "entity_name": entity_name, "records_extracted": 0,
        "records_ingested": 0, "api_calls_count": 0, "checkpoint_before": None,
        "checkpoint_after": None, "execution_time_sec": int(time.time() - ingest_start),
        "status": "NO_NEW_DATA", "run_date": run_date, "env": env, "created_at": datetime.now()
    }]).write.mode("append").saveAsTable(f"{catalog}.{schema_log}.bronze_ingest_runs")
    dbutils.notebook.exit("NO_NEW_DATA")

enriched = []
ts = datetime.now()
dt = ts.date()

for item in data_list:
    pjson = json.dumps(item, ensure_ascii=False)
    enriched.append({
        "payload": pjson,
        "record_id": str(item.get("id", "unknown")),
        "record_hash": hashlib.sha256(pjson.encode('utf-8')).hexdigest(),
        "load_id": load_id, "ingestion_timestamp": ts, "ingestion_date": dt,
        "source_endpoint": "/proposicoes", "env": env
    })

schema = StructType([
    StructField("payload", StringType()), StructField("record_id", StringType()),
    StructField("record_hash", StringType()), StructField("load_id", StringType()),
    StructField("ingestion_timestamp", TimestampType()), StructField("ingestion_date", DateType()),
    StructField("source_endpoint", StringType()), StructField("env", StringType())
])

df_bronze = spark.createDataFrame(enriched, schema=schema)
print(f"✅ DataFrame: {df_bronze.count()}")
print("="*70)

## 💾 Bloco 7: Salvar

In [0]:
print("💾 Salvando...\n")
try:
    df_bronze.write.format("delta").mode("append").partitionBy("ingestion_date").saveAsTable(table_bronze)
    print(f"✅ {num_records} salvos")
    ingest_success = True
    ingest_error = None
except Exception as e:
    ingest_success = False
    ingest_error = str(e)
    raise
finally:
    total_time = int(time.time() - ingest_start)
    print(f"⏱️ {total_time}s")
    print("="*70)

## 🔖 Bloco 8: Atualizar Checkpoint

In [0]:
print("🔖 Checkpoint...\n")
max_id = df_bronze.selectExpr("CAST(record_id AS INT) as id_int").agg({"id_int": "max"}).first()[0]

if max_id:
    spark.sql(f"""
        MERGE INTO {catalog}.{schema_ops}.checkpoints AS target
        USING (SELECT '/proposicoes' AS source_endpoint, 'last_id' AS checkpoint_type,
                      '{max_id}' AS checkpoint_value, current_timestamp() AS updated_at) AS source
        ON target.source_endpoint = source.source_endpoint AND target.checkpoint_type = source.checkpoint_type
        WHEN MATCHED THEN UPDATE SET target.checkpoint_value = source.checkpoint_value, target.updated_at = source.updated_at
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"✅ Checkpoint: {max_id}")
print("="*70)

## 📝 Bloco 9: Logging

In [0]:
print("📝 Log...\n")
api_calls = spark.sql(f"SELECT COUNT(*) FROM {catalog}.{schema_ops}.ingestion_requests WHERE load_id = '{load_id}' AND entity_name = '{entity_name}'").first()[0]

spark.createDataFrame([{
    "load_id": load_id, "entity_name": entity_name,
    "records_extracted": num_records, "records_ingested": num_records, "api_calls_count": api_calls,
    "checkpoint_before": str(checkpoint_value) if checkpoint_value else None,
    "checkpoint_after": str(max_id) if max_id else None,
    "execution_time_sec": total_time, "status": "SUCCESS",
    "error_message": None, "run_date": run_date, "env": env, "created_at": datetime.now()
}]).write.mode("append").saveAsTable(f"{catalog}.{schema_log}.bronze_ingest_runs")

print(f"✅ {num_records} registros, {api_calls} calls")
print("="*70)

## ✅ Bloco 10: Validação

In [0]:
print("✅ Validando...\n")
total = spark.sql(f"SELECT COUNT(*) FROM {table_bronze}").first()[0]
this_load = spark.sql(f"SELECT COUNT(*) FROM {table_bronze} WHERE load_id = '{load_id}'").first()[0]

print(f"📊 Total: {total:,} | Load: {this_load:,}")
print(f"   {'✅ OK' if this_load == num_records else '⚠️ DIV'}\n")

print("📊 Amostra:\n")
display(spark.sql(f"SELECT record_id, LEFT(record_hash, 12) as hash, ingestion_date, LEFT(payload, 70) as preview FROM {table_bronze} WHERE load_id = '{load_id}' LIMIT 5"))

print("\n" + "="*70)
print(f"\n🎉 Notebook 07_bronze_proposicoes concluído!")
print(f"\n🎯 Resumo:")
print(f"   • Tabela: {table_bronze}")
print(f"   • Total na tabela: {total:,} proposições")
print(f"   • Ingeridos agora: {this_load:,} proposições")
print(f"   • Checkpoint: {max_id}")
print(f"   • Status: SUCCESS ✅")